# pass@k Run Analysis

Loads samples.csv and pass_at_k.csv from a benchmark run. Supports model and temperature sweeps.


In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
import plotly.express as px

RUN_NAME = None  # e.g. "tinker_family_pairs_t07_k1_5_10_20"

candidates = [Path("results/runs"), Path("../results/runs")]
RUN_ROOT = next((p.resolve() for p in candidates if p.exists()), candidates[0].resolve())
PROJECT_ROOT = RUN_ROOT.parent.parent
EVAL_PIPELINE = PROJECT_ROOT / "eval-pipeline"
if str(EVAL_PIPELINE) not in sys.path:
    sys.path.insert(0, str(EVAL_PIPELINE))

if RUN_NAME:
    run = RUN_ROOT / RUN_NAME
else:
    runs = sorted([p for p in RUN_ROOT.glob("*") if p.is_dir()])
    run = runs[-1] if runs else None

assert run is not None and run.exists(), f"No run found under {RUN_ROOT}"
pass_at_k_csv = run / "pass_at_k.csv"
samples_csv = run / "samples.csv"
summary_json = run / "summary.json"

def rebuild_csvs_from_completed_model_jsons(run):
    from csv_exports import write_run_csvs
    from eval import _file_key, generate_summary

    config_path = run / "run_config.json"
    config = json.loads(config_path.read_text()) if config_path.exists() else {}
    provider = config.get("provider", "tinker")
    models = config.get("models", [])
    pass_k = config.get("pass_k", [1, 5, 10, 15, 20])

    all_results = {}
    for model in models:
        result_path = run / f"{_file_key(provider, model)}_results.json"
        if result_path.exists():
            all_results[f"{provider}/{model}"] = json.loads(result_path.read_text())

    if not all_results:
        for result_path in sorted(run.glob("*_results.json")):
            all_results[result_path.stem.removesuffix("_results")] = json.loads(result_path.read_text())

    assert all_results, f"No completed *_results.json files found in {run}"
    summary = generate_summary(all_results, execute_mode=True, metric_names=["pass_at_k", "accuracy"], pass_k=pass_k)
    summary_json.write_text(json.dumps(summary, indent=2))
    write_run_csvs(run, all_results, summary)
    return list(all_results)

if not pass_at_k_csv.exists() or not samples_csv.exists():
    rebuilt_models = rebuild_csvs_from_completed_model_jsons(run)
    print(f"Rebuilt partial CSVs from {len(rebuilt_models)} completed model result file(s).")

assert pass_at_k_csv.exists(), f"Missing pass@k CSV: {pass_at_k_csv}"
assert samples_csv.exists(), f"Missing samples CSV: {samples_csv}"
run


PosixPath('/mnt/c/Users/CB/OneDrive/Documents/UCLA Math Research/computational-algebra-llm/results/runs/tinker_family_pairs_t07_k1_5_10_20')

In [2]:
samples = pd.read_csv(samples_csv)
pass_at_k = pd.read_csv(pass_at_k_csv)
metric_cols = [c for c in pass_at_k.columns if c.startswith("pass@")]

aggregate = pass_at_k[pass_at_k["question_id"] == "__aggregate__"].copy()
aggregate


,model,question_id,temperature,num_samples,num_correct,num_attempted_samples,num_reference_failed_samples,pass@1,pass@5,pass@10,pass@20
116,tinker/Qwen/Qwen3-32B,__aggregate__,0.7,2900,1149,3900,1000,0.3962,0.5598,0.6078,0.6553
117,tinker/Qwen/Qwen3-30B-A3B-Instruct-2507,__aggregate__,0.7,2900,0,3900,1000,0.0000,0.0000,0.0000,0.0000


In [3]:
long = aggregate.melt(
    id_vars=["model", "temperature"],
    value_vars=metric_cols,
    var_name="metric",
    value_name="pass@k",
)
long["k"] = long["metric"].str.extract(r"pass@(\d+)").astype(int)
long["temperature_label"] = "T=" + long["temperature"].astype(str)
long = long.sort_values(["model", "temperature", "k"])

fig = px.line(
    long,
    x="k",
    y="pass@k",
    color="model",
    line_dash="temperature_label",
    markers=True,
    title="Pass@k by Model and Temperature",
    labels={"k": "k", "pass@k": "pass@k", "model": "Model", "temperature_label": "Temperature"},
)
fig.update_traces(line_shape="spline")
fig.update_layout(template="plotly_white", xaxis=dict(dtick=1), yaxis=dict(range=[0, 1]))
fig.show()


In [4]:
PLOT_K = max(int(c.split("@")[1]) for c in metric_cols)
temp_df = aggregate.sort_values(["model", "temperature"]).copy()

fig = px.line(
    temp_df,
    x="temperature",
    y=f"pass@{PLOT_K}",
    color="model",
    line_dash="model",
    markers=True,
    title=f"pass@{PLOT_K} by Temperature",
    labels={"temperature": "temperature", f"pass@{PLOT_K}": "pass@k", "model": "Model"},
)
fig.update_traces(line_shape="spline")
fig.update_layout(template="plotly_white", yaxis=dict(range=[0, 1]))
fig.show()


In [5]:
by_question = pass_at_k[pass_at_k["question_id"] != "__aggregate__"].copy()
by_question.head()


,model,question_id,temperature,num_samples,num_correct,num_attempted_samples,num_reference_failed_samples,pass@1,pass@5,pass@10,pass@20
0,tinker/Qwen/Qwen3-32B,m2_001,0.7,50,22,50,0,0.44,0.9536,0.9987,1.0000
1,tinker/Qwen/Qwen3-32B,m2_002,0.7,50,36,50,0,0.72,0.9991,1.0000,1.0000
2,tinker/Qwen/Qwen3-32B,m2_003,0.7,50,38,50,0,0.76,0.9996,1.0000,1.0000
3,tinker/Qwen/Qwen3-32B,m2_005,0.7,50,5,50,0,0.10,0.4234,0.6894,0.9327
4,tinker/Qwen/Qwen3-32B,m2_006,0.7,50,9,50,0,0.18,0.6463,0.8909,0.9943


In [6]:
group_cols = ["model"] + (["temperature"] if "temperature" in samples.columns else [])
agg_spec = {
    "samples": ("correct", "size"),
    "correct": ("correct", "sum"),
    "errors": ("error", lambda s: s.notna().sum()),
    "avg_total_tokens": ("total_tokens", "mean"),
}
if "oracle_raw_response" in samples.columns:
    agg_spec["oracle_calls"] = ("oracle_raw_response", lambda s: s.notna().sum())

sample_stats = samples.groupby(group_cols, dropna=False).agg(**agg_spec).reset_index()
sample_stats


,model,temperature,samples,correct,errors,avg_total_tokens,oracle_calls
0,tinker/Qwen/Qwen3-30B-A3B-Instruct-2507,0.7,3900,0,3900,NaN,0
1,tinker/Qwen/Qwen3-32B,0.7,3900,1149,299,1042.734518,1658
